# HW03 · Bài 3 · Multinomial Naïve Bayes

Dùng Bag of Words để đếm từ và Multinomial Naïve Bayes với làm trơn Laplace.

Phân loại cảm xúc trên dữ liệu IMDB của Stanford. Nhãn 0 là tiêu cực và nhãn 1 là tích cực. Cả năm notebook dùng chung mẫu 200 review lấy phân tầng từ tập test với seed 42 để so sánh.

## Nạp và tiền xử lý dữ liệu

Đọc review có nhãn từ tập IMDB gốc, sắp xếp theo tên tệp để việc lấy mẫu có thể lặp lại và bỏ thẻ HTML xuống dòng trước khi đưa review vào mô hình.

In [ ]:
import re
import shutil
import tarfile
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
SAMPLE_SIZE = 200
DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
ARCHIVE = Path("data/aclImdb_v1.tar.gz")
LABELS = {"neg": 0, "pos": 1}

if not ARCHIVE.exists():
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    partial = ARCHIVE.with_suffix(".tmp")
    try:
        with urlopen(DATA_URL, timeout=60) as response, partial.open("wb") as file:
            shutil.copyfileobj(response, file)
        partial.replace(ARCHIVE)
    finally:
        partial.unlink(missing_ok=True)


def load_imdb(splits):
    records = {split: [] for split in splits}
    with tarfile.open(ARCHIVE, "r|gz") as archive:
        for member in archive:
            parts = member.name.split("/")
            if not member.isfile() or len(parts) != 4 or parts[0] != "aclImdb":
                continue
            _, split, label, filename = parts
            if split not in records or label not in LABELS or not filename.endswith(".txt"):
                continue
            with archive.extractfile(member) as file:
                text = file.read().decode("utf-8")
            records[split].append((member.name, LABELS[label], text))

    datasets = {}
    for split, rows in records.items():
        if len(rows) != 25_000:
            raise ValueError(f"Tập {split} có {len(rows)} mẫu, dự kiến 25.000.")
        rows.sort(key=lambda row: row[0])
        datasets[split] = (
            [row[2] for row in rows],
            np.array([row[1] for row in rows], dtype=np.int32),
        )
    return datasets


def clean_text(text):
    return re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE).strip()

In [ ]:
data = load_imdb(("train", "test"))
X_train, y_train = data["train"]
X_test, y_test = data["test"]

shared_indices, _ = train_test_split(
    np.arange(len(y_test)), train_size=SAMPLE_SIZE,
    stratify=y_test, random_state=SEED,
)
shared_indices.sort()
print(f"IMDB test: {len(y_test):,} mẫu | Mẫu so sánh: {len(shared_indices)}")

## Độ đo đánh giá

Macro-F1 lấy trung bình F1 của hai nhãn. Accuracy và Macro-F1 được báo cáo trên thang từ 0 đến 1.

In [ ]:
def scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(
            y_true, y_pred, labels=[0, 1], average="macro", zero_division=0,
        ),
    }


def evaluate(model_name, predictions):
    return [
        {
            "Mô hình": model_name, "Tập đánh giá": "IMDB test", "Số mẫu": len(y_test),
            **scores(y_test, predictions),
        },
        {
            "Mô hình": model_name, "Tập đánh giá": "Mẫu so sánh",
            "Số mẫu": len(shared_indices),
            **scores(y_test[shared_indices], predictions[shared_indices]),
        },
    ]

## Huấn luyện và đánh giá

`Pipeline` học từ vựng và mô hình trên tập train.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    CountVectorizer(min_df=2),
    MultinomialNB(alpha=1.0),
)
model.fit([clean_text(text) for text in X_train], y_train)
predictions = model.predict([clean_text(text) for text in X_test])
display(pd.DataFrame(evaluate("BoW + Multinomial NB", predictions)).round(4))

## Tài liệu

[IMDB gốc](https://ai.stanford.edu/~amaas/data/sentiment/) · [MultinomialNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html)